Runtime → Change runtime type → **GPU** before running.

CNN (ResNet-50) counterpart of `vit_train.ipynb`. Same data, recipe and metric.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/xai-cnn-vit'
!mkdir -p $CKPT_DIR

Run from the `ViT/` folder of the repo (the `src` package lives there).

In [ ]:
# Replace with your repo URL. For a private repo use a GitHub token.
REPO = 'https://github.com/hsfcoulibaly/Explanability-Methods.git'
!git clone -q $REPO /content/repo || (cd /content/repo && git pull -q)
%cd /content/repo/ViT
!pip install -q timm captum

## Fine-tune (CNN)
Start with 5 epochs to confirm it works, then rerun with 10 / 20 / 30 (change `EPOCHS`). Checkpoint saves to Drive whenever test accuracy improves.

In [ ]:
EPOCHS = 5
!python -m src.train --arch cnn --epochs $EPOCHS --lr 3e-4 --batch-size 128 --out $CKPT_DIR/cnn_{EPOCHS}ep.pt

## Check the checkpoint + Grad-CAM / IG on a few images

In [ ]:
import torch, matplotlib.pyplot as plt
from src.models import load_checkpoint
from src.data import get_eval_subset, denormalize
from src.explain import gradcam, integrated_gradients

device = 'cuda'
model, ckpt = load_checkpoint('cnn', f'{CKPT_DIR}/cnn_{EPOCHS}ep.pt', device)
print('test acc:', ckpt['test_acc'], 'epoch:', ckpt['epoch'])

x, y = next(iter(get_eval_subset(batch_size=6)))
x = x.to(device)
cam, pred = gradcam(model, x)
ig, _ = integrated_gradients(model, x)
maps = [('Grad-CAM', cam), ('IG', ig)]

fig, ax = plt.subplots(1 + len(maps), 6, figsize=(15, 2.6 * (1 + len(maps))))
for i in range(6):
    ax[0, i].imshow(denormalize(x[i]).permute(1, 2, 0).cpu())
    ax[0, i].set_title(f'y={y[i].item()} pred={pred[i].item()}', fontsize=9)
    for r, (name, m) in enumerate(maps, 1):
        ax[r, i].imshow(denormalize(x[i]).permute(1, 2, 0).cpu())
        ax[r, i].imshow(m[i].cpu(), cmap='jet', alpha=0.5)
        ax[r, 0].set_ylabel(name)
for a in ax.flat: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Faithfulness scores (CNN)
`max_batches=10` → 160 images for a quick check; set `None` for the full 1000. IG is slow (32 steps/image).

In [ ]:
from src.metrics import evaluate_explainer
loader = get_eval_subset(batch_size=16)
results = {}
for name, fn in [('gradcam', gradcam), ('ig', integrated_gradients)]:
    results[name] = evaluate_explainer(model, loader, fn, device, max_batches=10)
    print('cnn +', name, results[name])

## Full evaluation (writes/merges `results.json` on Drive)
Remove `--max-batches 4` for the full 1,000 images. Run the `--removal mean` line too (ablation).

In [ ]:
CK = f'{CKPT_DIR}/cnn_{EPOCHS}ep.pt'
!python -m src.run_eval --arch cnn --ckpt {CK} --out $CKPT_DIR/results.json --max-batches 4
# !python -m src.run_eval --arch cnn --ckpt {CK} --out $CKPT_DIR/results.json --removal mean

## Randomization sanity check (cascading, top-down)

In [ ]:
!python -m src.sanity --arch cnn --ckpt {CK} --out $CKPT_DIR/sanity_cnn.json --n-images 64